In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

project_dir = Path.cwd().parent
csv_path = project_dir / "data" / "processed" / "online_retail_prepared.csv"
db_path = project_dir / "data" / "processed" / "online_retail.db"

print("CSV exists:", csv_path.exists())
print("Database path:", db_path)

CSV exists: True
Database path: D:\online-retail-analytics\data\processed\online_retail.db


In [2]:
conn = sqlite3.connect(db_path)

first_chunk = True

for chunk in pd.read_csv(
    csv_path,
    chunksize=100_000,
    dtype={
        "invoice": "string",
        "stock_code": "string",
        "description": "string",
        "country": "string",
        "transaction_category": "string",
        "review_status": "string",
    },
):
    chunk.to_sql(
        "online_retail",
        conn,
        if_exists="replace" if first_chunk else "append",
        index=False,
    )
    first_chunk = False

print("Import complete.")

Import complete.


In [3]:
pd.read_sql_query(
    "SELECT COUNT(*) AS total_rows FROM online_retail;",
    conn,
)

,total_rows
0,1044848


In [4]:
pd.read_sql_query(
    "PRAGMA table_info(online_retail);",
    conn,
)

,cid,name,type,notnull,dflt_value,pk
0,0,invoice,TEXT,0,None,0
1,1,stock_code,TEXT,0,None,0
2,2,description,TEXT,0,None,0
3,3,quantity,INTEGER,0,None,0
4,4,invoice_date,TEXT,0,None,0
5,5,price,REAL,0,None,0
6,6,customer_id,REAL,0,None,0
7,7,country,TEXT,0,None,0
8,8,line_value,REAL,0,None,0
9,9,is_cancellation_invoice,INTEGER,0,None,0


In [5]:
pd.read_sql_query("""
SELECT
    transaction_category,
    COUNT(*) AS row_count,
    ROUND(SUM(line_value), 2) AS total_line_value
FROM online_retail
GROUP BY transaction_category
ORDER BY total_line_value;
""", conn)

,transaction_category,row_count,total_line_value
0,cancellation_style_invoice,19165,-1465303.66
1,bad_debt_adjustment,6,-147614.08
2,negative_quantity_non_c_invoice,3393,0.00
3,other_or_ordinary,1022284,20522679.86


In [6]:
pd.read_sql_query("""
SELECT
    SUBSTR(invoice_date, 1, 4) AS sales_year,
    COUNT(*) AS row_count,
    ROUND(SUM(line_value), 2) AS total_line_value
FROM online_retail
GROUP BY SUBSTR(invoice_date, 1, 4)
ORDER BY sales_year;
""", conn)

,sales_year,row_count,total_line_value
0,2009,45228,799847.11
1,2010,500191,9111106.09
2,2011,499429,8998808.91


In [7]:
pd.read_sql_query("""
SELECT
    CASE
        WHEN customer_id IS NULL THEN 'Missing customer ID'
        ELSE 'Customer ID present'
    END AS customer_status,
    COUNT(*) AS row_count,
    ROUND(SUM(line_value), 2) AS total_line_value
FROM online_retail
GROUP BY customer_status;
""", conn)

,customer_status,row_count,total_line_value
0,Customer ID present,809561,16343669.04
1,Missing customer ID,235287,2566093.08


In [8]:
pd.read_sql_query("""
SELECT
    COUNT(*) AS extra_duplicate_rows,
    ROUND(SUM(line_value), 2) AS signed_line_value,
    ROUND(SUM(ABS(line_value)), 2) AS absolute_line_value
FROM online_retail
WHERE is_duplicate_extra = 1;
""", conn)

,extra_duplicate_rows,signed_line_value,absolute_line_value
0,11812,54228.42,60734.52


In [9]:
pd.read_sql_query("""
SELECT
    transaction_category,
    COUNT(*) AS row_count,
    ROUND(SUM(line_value), 2) AS total_line_value,
    SUM(CASE WHEN quantity < 0 THEN 1 ELSE 0 END)
        AS negative_quantity_rows
FROM online_retail
GROUP BY transaction_category
ORDER BY row_count DESC;
""", conn)

,transaction_category,row_count,total_line_value,negative_quantity_rows
0,other_or_ordinary,1022284,20522679.86,0
1,cancellation_style_invoice,19165,-1465303.66,19164
2,negative_quantity_non_c_invoice,3393,0.00,3393
3,bad_debt_adjustment,6,-147614.08,0


In [10]:
import sqlite3
from pathlib import Path

project_dir = Path.cwd().parent
db_path = project_dir / "data" / "processed" / "online_retail.db"

with sqlite3.connect(db_path) as conn:
    print(pd.read_sql_query("""
        SELECT transaction_category,
               COUNT(*) AS row_count,
               ROUND(SUM(line_value), 2) AS total_line_value
        FROM online_retail
        GROUP BY transaction_category
        ORDER BY transaction_category
    """, conn))

              transaction_category  row_count  total_line_value
0              bad_debt_adjustment          6        -147614.08
1       cancellation_style_invoice      19165       -1465303.66
2  negative_quantity_non_c_invoice       3393              0.00
3                other_or_ordinary    1022284       20522679.86
